Import all functions

In [5]:
from GastruloidKit.bins import *
from GastruloidKit.distributions import *
from GastruloidKit.detection import *
from GastruloidKit.intensity_bins import *
from GastruloidKit.preprocessing import *
from GastruloidKit.visualizechannels import *

For clear step by step explanation look at the README.md file!
## Part I: Preprocessing

In [ ]:
# 1. EXPERIMENT DETAILS SET UP ---------------------------------
wt = "WT" # this should be WildType 
mutant = "ND6" # change this to your mutant name 
conditions = [wt, mutant]

repeats = [1,2,3] # set this to whichever repeat you want to focus on, it could be [1] only or [1,3] only, etc. any combination you want. 

markers = ["DAPI", "SOX2", "BRA", "GATA3"] # change this to your markers 
ref_marker = "DAPI" # your reference marker / marker to mark cells in general

# change this to your markers and how theyre arranged on the CZI channels - look at ur images on ImageJ when assigning this
channel_folders = { 
        0: 'DAPI',
        1: 'SOX2',
        2: 'GATA3',
        3: 'BRA'
    }
# in the example above, channel 0 is DAPI, channel 1 is SOX2 and so on..
# -------------------------------------------------------------
# 2. DATA ANALYSIS SET UP -------------------------------------
directory = "CHIP_REPEATS" # directory in which you saved your .czi images saved in the format explained in the README.md file

marker_colors = { # This dictionary will determine what marker is plotted in what color. This is entirely up to you!
    'SOX2': '#00bcd4', # cyan
    'BRA': '#ffeb3b', # yellow
    'GATA3': '#9c27b0', # magenta
    'DAPI': "#ffffffff", # white
    'psmad159': "#ff0000ff" # red
}

num_bins = 15 # set how many bins you want; in this example I set 15
gastruloid_radius = 110 # the radius of the gastruloid (will be the outermost circle); in this example I set 110 pixels

In [ ]:
czi_to_tiff(directory) # convert CZI into Tiff

In [ ]:
check_dimensions(directory) # it should show your scaled images na dmasks have dimensions ({number of channels}, 7800, 7800)

In [ ]:
split_into_channels(directory, repeats, conditions, channel_folders) # split tiffs into channels 

In [ ]:
grid_split(directory, markers, conditions, repeats) # split into 676 boxes using a 26 x 26 grid.

In [ ]:
select_gastruloids(directory, ref_marker, markers, conditions, repeats) # opens pop-up for selection, makes the new folder containing only selected boxes. 

## Part II: Radial Bin Analysis

In [ ]:
# First time running it: run it with adjusting True and Loading False 
bin_setting(directory, repeats, conditions, markers, gastruloid_radius, num_bins, adjusting=True, loading=False)

In [ ]:
# When adjusting for gastruloid radius set adjusting True and Loading True
bin_setting(directory, repeats, conditions, markers, gastruloid_radius, num_bins, adjusting=True, loading=True)

In [ ]:
# when you are not adjusting for gastruloid radius set adjusting False and loading True. 
bin_setting(directory, repeats, conditions, markers, gastruloid_radius, num_bins, adjusting=False, loading=True)

# Remember loading is always False the first time you run it. Always True the next time you do to make sure it runs faster as making binary masks take quite awhile!!

In [ ]:
make_GATA3_filter(directory, repeats, conditions) # makes the GATA3 filter and saves it in a folder called GATA3filter 

In [ ]:
# measure raw intensities of all markers 
get_rawintensities(directory, repeats, conditions, markers, gastruloid_radius, num_bins)

# normalize intensities of all markers using your reference marker (DAPI in my case)
normalize_intensities(directory, repeats, conditions, markers, ref_marker, num_bins)

## Part III: Plotting

In [ ]:
# make plots 
plot_gastruloidprofiles(directory, repeats, conditions, markers, ref_marker, num_bins, marker_colors)

In [ ]:
get_distributions(directory, repeats, conditions, markers) # get raw whole intensity distirbutions of all markers including DAPI
DAPIintensity_split_profiles(directory, repeats, conditions, num_bins, marker_colors) # split profiles by DAPI Intensity

# to run the one below, you need to run get_distributions first above as getting DAPI center size relies on DAPI intensity distirbution
get_DAPIcenter_distributions(directory, repeats, conditions, num_bins) # get DAPI center size distribution
DAPIcenter_split_profiles(directory, repeats, conditions, num_bins, marker_colors) # split profiles by DAPI center size

In [ ]:
# Set the details of which gastruloid you want to plot above -----------
chosen_condition = 'WT'
chosen_repeat = 1
chosen_id = 275 # look at the ID in boxes_tiff_selected folder to choose from
# ---------------------------------------------------------------------------
channels_plot_any(chosen_id, directory, chosen_repeat, chosen_condition, markers, marker_colors, ref_marker, include_ref_marker=False) # makes a plot like the one above
# choose whether to include your nuclear counterstain / DAPI from the merged image. 

markers_pair=("DAPI", "BRA") # 2 chosen markers for one that does pairs (because why not)
channels_plot_pair(chosen_id, directory, chosen_repeat, chosen_condition, markers_pair, marker_colors) # same as above but only for 2 chosen markers 